In [143]:
import pandas as pd
from pathlib import Path
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials
from gspread_formatting import *

In [157]:
DIR_PATH = Path("/home/cognify/Downloads/SEC_Funds/Cascade_Private_Capital_Fund")
CONSOLIDATED_CSV_PATH = Path(DIR_PATH, "CPCF_Output/ea0264559-01_ncsrs.csv")
ANALYSIS_PATH = Path(DIR_PATH, "analysis")
ANALYSIS_PATH.mkdir(exist_ok=True)

DIR_PATH.exists(), CONSOLIDATED_CSV_PATH.exists(), ANALYSIS_PATH.exists()

(True, True, True)

In [158]:
df_investment = pd.read_csv(CONSOLIDATED_CSV_PATH)
df_investment.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   portfolio_company         244 non-null    str    
 1   initial_acquisition_date  242 non-null    str    
 2   geographic_region         242 non-null    str    
 3   fair_value                244 non-null    str    
 4   asset_class               0 non-null      float64
 5   principal_amount          6 non-null      float64
 6   shares_units              28 non-null     float64
 7   industry                  244 non-null    str    
 8   part                      244 non-null    str    
 9   table_index               244 non-null    int64  
 10  row_index                 244 non-null    int64  
 11  cost                      241 non-null    float64
dtypes: float64(4), int64(2), str(6)
memory usage: 23.0 KB


In [159]:
df_investment[df_investment["portfolio_company"].str.contains("Platte River", na=False)]

,portfolio_company,initial_acquisition_date,geographic_region,fair_value,asset_class,principal_amount,shares_units,industry,part,table_index,row_index,cost
130,"Platte River Equity III, L.P.",9/30/2024,North America,5907663,NaN,NaN,NaN,Secondary Fund Investments — 43.6%,ea0264559-01_ncsrs,7,24,4473486.0


In [160]:
df_investment.head(275)

,portfolio_company,initial_acquisition_date,geographic_region,fair_value,asset_class,principal_amount,shares_units,industry,part,table_index,row_index,cost
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,7183938,NaN,NaN,NaN,Primary Fund Investments — 28.2%,ea0264559-01_ncsrs,1,2,4443432.0
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,40164236,NaN,NaN,NaN,Primary Fund Investments — 28.2%,ea0264559-01_ncsrs,1,3,30000000.0
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,7452207,NaN,NaN,NaN,Primary Fund Investments — 28.2%,ea0264559-01_ncsrs,1,4,6073055.0
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,79606439,NaN,NaN,68333.0,Primary Fund Investments — 28.2%,ea0264559-01_ncsrs,1,5,70000000.0
4,"DFJ Growth V, L.P.",11/27/2024,North America,2208839,NaN,NaN,NaN,Primary Fund Investments — 28.2%,ea0264559-01_ncsrs,1,6,1775000.0
...,...,...,...,...,...,...,...,...,...,...,...,...
239,"Nader Upside 2 Sarl, EUR PIK Term Loan, 3 mo. ...",3/14/2024,Europe,6424011,NaN,5548541.0,NaN,Credit Co-Investments — 2.1%,ea0264559-01_ncsrs,13,25,6004819.0
240,"Symbiotic Capital EB Fund, L.P.",3/7/2024,North America,5744363,NaN,NaN,NaN,Credit Co-Investments — 2.1%,ea0264559-01_ncsrs,13,26,5061912.0
241,Dawson Partners Portflio Finance Evergreen (Ag...,8/25/2025,North America,110925,NaN,NaN,NaN,Private Equity — 0.0%,ea0264559-01_ncsrs,13,30,100.0
242,Cliffwater Corporate Lending Fund — Class I,NaN,NaN,101412429,NaN,NaN,9416196.0,Registered Funds — 2.4%,ea0264559-01_ncsrs,13,33,100000000.0


In [161]:
# asset_list = list(df_asset_class_level_analysis["Asset Class"])
# name = [v.split("—")[0].strip() for v in asset_list]
# perc = [v.split("—")[1] for v in asset_list]

# df_asset_class_level_analysis = df_asset_class_level_analysis.drop(columns=["Asset Class"])
# df_asset_class_level_analysis.insert(0, "Assets", name)
# df_asset_class_level_analysis.insert(1, "percentage", perc)

# df_asset_class_level_analysis

In [162]:
column_mapping = {
    "portfolio_company": "Portfolio Company",
    "initial_acquisition_date": "Initial Acquisition Date",
    "geographic_region": "Geographic Region",
    "shares_units": "Shares Units",
    "cost": "Cost",
    "fair_value": "Fair Value",
    "first_acquisition_date": "First Acquisition Date",
    "asset_class": "Asset Class",
    "industry": "Industry",
    "principal_amount": "Principal Amount",
    "profit": "Profit"
}

desired_order = [
    "Portfolio Company",
    "Initial Acquisition Date",
    "Geographic Region",
    "Principal Amount",
    "Shares Units",
    "Cost",
    "Fair Value",
    "Profit",
    "First Acquisition Date",
    "Asset Class",
    "Industry"
]
df_investment_formatted = df_investment.rename(columns=column_mapping).reindex(columns=desired_order)
df_investment_formatted

,Portfolio Company,Initial Acquisition Date,Geographic Region,Principal Amount,Shares Units,Cost,Fair Value,Profit,First Acquisition Date,Asset Class,Industry
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,NaN,NaN,4443432.0,7183938,NaN,NaN,NaN,Primary Fund Investments — 28.2%
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,NaN,NaN,30000000.0,40164236,NaN,NaN,NaN,Primary Fund Investments — 28.2%
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,NaN,NaN,6073055.0,7452207,NaN,NaN,NaN,Primary Fund Investments — 28.2%
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,NaN,68333.0,70000000.0,79606439,NaN,NaN,NaN,Primary Fund Investments — 28.2%
4,"DFJ Growth V, L.P.",11/27/2024,North America,NaN,NaN,1775000.0,2208839,NaN,NaN,NaN,Primary Fund Investments — 28.2%
...,...,...,...,...,...,...,...,...,...,...,...
239,"Nader Upside 2 Sarl, EUR PIK Term Loan, 3 mo. ...",3/14/2024,Europe,5548541.0,NaN,6004819.0,6424011,NaN,NaN,NaN,Credit Co-Investments — 2.1%
240,"Symbiotic Capital EB Fund, L.P.",3/7/2024,North America,NaN,NaN,5061912.0,5744363,NaN,NaN,NaN,Credit Co-Investments — 2.1%
241,Dawson Partners Portflio Finance Evergreen (Ag...,8/25/2025,North America,NaN,NaN,100.0,110925,NaN,NaN,NaN,Private Equity — 0.0%
242,Cliffwater Corporate Lending Fund — Class I,NaN,NaN,NaN,9416196.0,100000000.0,101412429,NaN,NaN,NaN,Registered Funds — 2.4%


### Dumping to GS

In [163]:
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    "service_account.json",
    scopes=SCOPES
)

client = gspread.authorize(creds)

In [164]:
df_map = {
    # "Summary": df_summary,
    "All Investments": df_investment_formatted,
    # "Asset Level Analysis": df_asset_class_level_analysis,
    # "Company Level Analysis": df_company_level_analysis
}

In [165]:
spreadsheet = client.open("CPCF-ea0264559-01_ncsrs")

In [166]:
for tab_name, df_analysis in df_map.items():
    try:
        worksheet = spreadsheet.worksheet(tab_name)
        worksheet.clear()
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=tab_name,
            rows=max(len(df_analysis) + 10, 100),
            cols=max(len(df_analysis.columns) + 10, 20)
        )

    display(df_analysis)
    set_with_dataframe(worksheet, df_analysis)
    
    header_fmt = CellFormat(
        backgroundColor=Color(0.85, 0.90, 1.0),
        textFormat=TextFormat(bold=True)
    )

    format_cell_range(
        worksheet,
        "1:1",
        header_fmt
    )

    worksheet.freeze(rows=1)
    worksheet.columns_auto_resize(0, len(df_analysis.columns))

,Portfolio Company,Initial Acquisition Date,Geographic Region,Principal Amount,Shares Units,Cost,Fair Value,Profit,First Acquisition Date,Asset Class,Industry
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,NaN,NaN,4443432.0,7183938,NaN,NaN,NaN,Primary Fund Investments — 28.2%
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,NaN,NaN,30000000.0,40164236,NaN,NaN,NaN,Primary Fund Investments — 28.2%
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,NaN,NaN,6073055.0,7452207,NaN,NaN,NaN,Primary Fund Investments — 28.2%
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,NaN,68333.0,70000000.0,79606439,NaN,NaN,NaN,Primary Fund Investments — 28.2%
4,"DFJ Growth V, L.P.",11/27/2024,North America,NaN,NaN,1775000.0,2208839,NaN,NaN,NaN,Primary Fund Investments — 28.2%
...,...,...,...,...,...,...,...,...,...,...,...
239,"Nader Upside 2 Sarl, EUR PIK Term Loan, 3 mo. ...",3/14/2024,Europe,5548541.0,NaN,6004819.0,6424011,NaN,NaN,NaN,Credit Co-Investments — 2.1%
240,"Symbiotic Capital EB Fund, L.P.",3/7/2024,North America,NaN,NaN,5061912.0,5744363,NaN,NaN,NaN,Credit Co-Investments — 2.1%
241,Dawson Partners Portflio Finance Evergreen (Ag...,8/25/2025,North America,NaN,NaN,100.0,110925,NaN,NaN,NaN,Private Equity — 0.0%
242,Cliffwater Corporate Lending Fund — Class I,NaN,NaN,NaN,9416196.0,100000000.0,101412429,NaN,NaN,NaN,Registered Funds — 2.4%


In [33]:
for i  in df_investment['asset_class'].unique(): print(i)

Primary Fund Investments — 25.3%
Secondary Fund Investments — 40.4%
Equity Co-Investments — 31.5%
Credit Co-Investments — 3.2%
Private Equity — 0.0%
Registered Funds — 1.8%
Short-Term Investments — 4.6%


nan


,portfolio_company,initial_acquisition_date,geographic_region,cost,fair_value,asset_class,industry,part,table_index,row_index,shares_units,principal_amount
0,"Bertram Growth Capital IV-A, L.P.",1/7/2022,North America,1238736.0,4963620.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,2,NaN,NaN
1,BlackRock Secondaries & Liquidity Solutions II...,12/27/2024,North America,46000000.0,61789798.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,3,NaN,NaN
2,"BPOC Fund VI-A, L.P.",6/6/2025,North America,6324758.0,7742836.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,4,NaN,NaN
3,Dawson Portfolio Finance Evergreen LP,5/28/2024,North America,70000000.0,81968442.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,5,68333.0,NaN
4,"DFJ Growth V, L.P.",11/27/2024,North America,2475000.0,3683889.0,NaN,Primary Fund Investments — 25.3%,ea0289743-01_ncsr,1,6,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
276,"RCAF CCG, L.P.",12/29/2025,North America,70000000.0,70000000.0,NaN,Credit Co-Investments — 3.2%,ea0289743-01_ncsr,15,2,NaN,NaN
277,"Symbiotic Capital EB Fund, L.P.",3/7/2024,North America,5061912.0,5779029.0,NaN,Credit Co-Investments — 3.2%,ea0289743-01_ncsr,15,3,NaN,NaN
278,Dawson Partners Portfolio Finance Evergreen (A...,8/25/2025,North America,4784.0,93164.0,NaN,Private Equity — 0.0%,ea0289743-01_ncsr,15,7,NaN,NaN
279,Cliffwater Corporate Lending Fund — Class I,NaN,North America,100000000.0,99246704.0,NaN,Registered Funds — 1.8%,ea0289743-01_ncsr,15,11,NaN,9416196.0
